# Mumbai & Navi Mumbai AQI (2015–2020): Coordinate Integration + ML

This notebook uses the classic Kaggle **Air Quality Data in India (2015–2020)** files `station_day.csv` and `stations.csv`, plus the accompanying external station-coordinate file `station_coordinates_external.csv`.

**Important finding from the supplied files:** `stations.csv` has 13 Mumbai/Navi Mumbai station metadata rows, but `station_day.csv` has daily observations for the 10 Mumbai station IDs and **zero observations for the three Navi Mumbai station IDs (MH017, MH018, MH019)**. Adding coordinates does not create historical measurements. The notebook will report this clearly and will not invent Navi Mumbai observations.

Coordinates in the external file are mixed provenance: several are published station coordinates; others are approximate locality coordinates and marked as such. For a graded spatial analysis, verify approximate points against official MPCB coordinates or station addresses before presenting them as exact locations.

The ML target is the observed `AQI`; predictors are pollutant measurements. This is same-day AQI estimation, not future AQI forecasting.

## 1. Install and import libraries

In [ ]:
# If needed, run once:
# %pip install pandas numpy matplotlib seaborn scikit-learn

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path(".")  # Change this if files are stored elsewhere
OUTPUT_DIR = Path("aqi_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

STATION_DAY_FILE = DATA_DIR / "station_day.csv"
STATIONS_FILE = DATA_DIR / "stations.csv"
COORDINATES_FILE = DATA_DIR / "station_coordinates_external.csv"

for file in [STATION_DAY_FILE, STATIONS_FILE, COORDINATES_FILE]:
    print(file, "exists:", file.exists())
    if not file.exists():
        raise FileNotFoundError(f"Missing {file}. Place all three CSVs beside this notebook or update DATA_DIR.")


## 2. Load the Kaggle files and external coordinate file

In [ ]:
daily_raw = pd.read_csv(STATION_DAY_FILE)
stations_raw = pd.read_csv(STATIONS_FILE)
coords_raw = pd.read_csv(COORDINATES_FILE)

print("station_day:", daily_raw.shape)
print("stations:", stations_raw.shape)
print("external coordinates:", coords_raw.shape)

print("\nStation-day columns:", daily_raw.columns.tolist())
print("Station metadata columns:", stations_raw.columns.tolist())
print("Coordinate columns:", coords_raw.columns.tolist())

display(daily_raw.head())
display(stations_raw[stations_raw["City"].astype(str).str.contains("Mumbai", case=False, na=False)])
display(coords_raw)


## 3. Standardize names and validate coordinate data

In [ ]:
def standardize_columns(df):
    out = df.copy()
    out.columns = (
        out.columns.astype(str).str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

daily = standardize_columns(daily_raw)
stations = standardize_columns(stations_raw)
coords = standardize_columns(coords_raw)

required_daily = {"stationid", "date", "aqi"}
required_stations = {"stationid", "stationname", "city", "state"}
required_coords = {"stationid", "latitude", "longitude"}

if not required_daily.issubset(daily.columns):
    raise ValueError(f"station_day.csv missing columns: {required_daily - set(daily.columns)}")
if not required_stations.issubset(stations.columns):
    raise ValueError(f"stations.csv missing columns: {required_stations - set(stations.columns)}")
if not required_coords.issubset(coords.columns):
    raise ValueError(f"Coordinate CSV missing columns: {required_coords - set(coords.columns)}")

coords["latitude"] = pd.to_numeric(coords["latitude"], errors="coerce")
coords["longitude"] = pd.to_numeric(coords["longitude"], errors="coerce")

bad_coords = coords[
    coords["latitude"].isna() | coords["longitude"].isna()
    | ~coords["latitude"].between(-90, 90)
    | ~coords["longitude"].between(-180, 180)
]
if len(bad_coords):
    raise ValueError("Invalid coordinates found:\n" + bad_coords.to_string(index=False))

if coords["stationid"].duplicated().any():
    raise ValueError("Duplicate StationId values in coordinate file. Resolve before joining.")

print("Coordinate validation passed.")


## 4. Integrate station metadata and coordinates by StationId

In [ ]:
# Keep only the stations that the original metadata identifies as Mumbai or Navi Mumbai.
target_stations = stations[
    stations["city"].astype(str).str.strip().str.lower().isin(["mumbai", "navi mumbai"])
].copy()

# Coordinate file is matched by stable StationId, not fuzzy name matching.
station_geo = target_stations.merge(
    coords,
    on="stationid",
    how="left",
    suffixes=("_metadata", "_coordinate"),
    indicator=True
)

print("Target station metadata rows:", len(target_stations))
print("Stations with coordinate matches:", int((station_geo["_merge"] == "both").sum()))
print("Stations without coordinate matches:", int((station_geo["_merge"] != "both").sum()))

if station_geo[["latitude", "longitude"]].isna().any().any():
    display(station_geo[station_geo["latitude"].isna() | station_geo["longitude"].isna()])
    raise ValueError("Some Mumbai/Navi Mumbai stations lack coordinates. Fix coordinate CSV before proceeding.")

# Preserve station names and city labels from the original Kaggle metadata.
station_geo = station_geo.rename(columns={
    "stationname_metadata": "stationname",
    "city_metadata": "city"
})
station_geo["city"] = station_geo["city"].astype(str).str.strip()

display(station_geo[[
    "stationid", "stationname", "city", "state",
    "latitude", "longitude", "coordinate_notes", "coordinate_source_type"
]])


## 5. Filter observations to the project stations and validate the date range

In [ ]:
daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
daily["aqi"] = pd.to_numeric(daily["aqi"], errors="coerce")

# Inner join means only station IDs present in the verified Mumbai/Navi Mumbai station table survive.
project = daily.merge(
    station_geo,
    on="stationid",
    how="inner",
    suffixes=("", "_station")
)

project = project[project["date"].between("2015-01-01", "2020-12-31")].copy()
project = project.drop_duplicates()

print("Project observations:", len(project))
print("Date coverage:", project["date"].min(), "to", project["date"].max())
print("\nObservations by city:")
city_coverage = project.groupby("city").agg(
    rows=("stationid", "size"),
    stations=("stationid", "nunique"),
    first_date=("date", "min"),
    last_date=("date", "max")
)
display(city_coverage)

print("\nStation coverage, including zero-row stations:")
station_counts = (
    project.groupby(["stationid", "stationname", "city"])
    .size().rename("observation_rows").reset_index()
)
coverage_all = station_geo[["stationid", "stationname", "city"]].merge(
    station_counts, on=["stationid", "stationname", "city"], how="left"
)
coverage_all["observation_rows"] = coverage_all["observation_rows"].fillna(0).astype(int)
display(coverage_all.sort_values(["city", "stationid"]))

navi_rows = int((project["city"].str.lower() == "navi mumbai").sum())
if navi_rows == 0:
    print(
        "\nDATA LIMITATION: No Navi Mumbai observations exist in the supplied station_day.csv "
        "for these station IDs. Coordinates are successfully joined, but cannot supply missing historical readings. "
        "The model below will run on the available Mumbai observations only. For a valid two-city comparison, "
        "integrate a second historical AQI source for Navi Mumbai."
    )


## 6. Save the joined geographic data before cleaning

In [ ]:
joined_file = OUTPUT_DIR / "mumbai_navi_mumbai_geocoded_before_cleaning.csv"
project.to_csv(joined_file, index=False)
print("Saved:", joined_file.resolve())


## 7. Clean data and engineer features

In [ ]:
pollutant_cols = [
    "pm2_5", "pm10", "no", "no2", "nox", "nh3",
    "co", "so2", "o3", "benzene", "toluene", "xylene"
]
pollutant_cols = [c for c in pollutant_cols if c in project.columns]

clean = project.copy()
rows_before = len(clean)

clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
for col in ["aqi"] + pollutant_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

clean = clean.drop_duplicates()
clean = clean.dropna(subset=["date", "aqi"])
clean = clean[clean["aqi"] >= 0].copy()

clean["year"] = clean["date"].dt.year
clean["month"] = clean["date"].dt.month
clean["month_name"] = clean["date"].dt.month_name()
clean["quarter"] = clean["date"].dt.quarter
clean["day_of_week"] = clean["date"].dt.dayofweek
clean["day_name"] = clean["date"].dt.day_name()
clean["weekend"] = (clean["date"].dt.dayofweek >= 5).astype(int)

def season_from_month(m):
    if m in [12, 1, 2]: return "Winter"
    if m in [3, 4, 5]: return "Summer"
    if m in [6, 7, 8, 9]: return "Monsoon"
    if m in [10, 11]: return "Post-Monsoon"
    return np.nan

def aqi_category(x):
    if pd.isna(x): return np.nan
    if x <= 50: return "Good"
    if x <= 100: return "Satisfactory"
    if x <= 200: return "Moderate"
    if x <= 300: return "Poor"
    if x <= 400: return "Very Poor"
    return "Severe"

clean["season"] = clean["month"].apply(season_from_month)
clean["aqi_category"] = clean["aqi"].apply(aqi_category)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(clean))
print("Removed rows:", rows_before - len(clean))
display(clean[["date", "stationid", "stationname", "city", "latitude", "longitude", "aqi", "aqi_category"]].head())
display(clean[["aqi"] + pollutant_cols].isna().sum().to_frame("missing_values"))


## 8. EDA: city and time trends

In [ ]:
display(clean.groupby("city")["aqi"].agg(["count", "mean", "median", "min", "max"]).round(2))

if len(clean):
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=clean, x="city", y="aqi")
    plt.title("AQI Distribution by City")
    plt.tight_layout()
    plt.show()

    yearly = clean.groupby(["year", "city"], as_index=False)["aqi"].mean()
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=yearly, x="year", y="aqi", hue="city", marker="o")
    plt.title("Annual Mean AQI by City")
    plt.tight_layout()
    plt.show()

    monthly = clean.groupby(["month", "city"], as_index=False)["aqi"].mean()
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=monthly, x="month", y="aqi", hue="city", marker="o")
    plt.title("Monthly Mean AQI by City")
    plt.xticks(range(1, 13))
    plt.tight_layout()
    plt.show()


## 9. Train Random Forest Regression

Predictors are pollutant measurements. Latitude and longitude are used for geographic selection and Power BI mapping, **not** as ML predictors. The model uses a chronological 80/20 split and imputes missing predictors using training data only.

Because the supplied Kaggle files have no Navi Mumbai observations, the current model will be trained on Mumbai observations only. Do not interpret its scores as a two-city model.

In [ ]:
if not pollutant_cols:
    raise ValueError("No pollutant columns found for modelling.")

model_data = clean.sort_values("date").reset_index(drop=True).copy()
split_at = int(len(model_data) * 0.80)
if split_at < 2 or len(model_data) - split_at < 1:
    raise ValueError("Not enough rows to create a chronological 80/20 split.")

train_df = model_data.iloc[:split_at].copy()
test_df = model_data.iloc[split_at:].copy()

X_train = train_df[pollutant_cols]
y_train = train_df["aqi"]
X_test = test_df[pollutant_cols]
y_test = test_df["aqi"]

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred) if len(y_test) > 1 else np.nan

metrics = pd.DataFrame([
    {"Metric": "MAE", "Value": mae},
    {"Metric": "RMSE", "Value": rmse},
    {"Metric": "R2", "Value": r2},
    {"Metric": "Training_Rows", "Value": len(train_df)},
    {"Metric": "Test_Rows", "Value": len(test_df)},
    {"Metric": "Navi_Mumbai_Observation_Rows", "Value": int((clean["city"].str.lower() == "navi mumbai").sum())}
])
display(metrics)
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)


In [ ]:
# Actual vs predicted plot
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.5)
lo = min(float(y_test.min()), float(np.min(y_pred)))
hi = max(float(y_test.max()), float(np.max(y_pred)))
plt.plot([lo, hi], [lo, hi], linestyle="--")
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Actual vs Predicted AQI — Test Set")
plt.tight_layout()
plt.show()

# Error distribution
errors = y_test.to_numpy() - y_pred
plt.figure(figsize=(9, 5))
sns.histplot(errors, bins=35, kde=True)
plt.axvline(0, linestyle="--")
plt.xlabel("Actual AQI - Predicted AQI")
plt.title("Prediction Error Distribution")
plt.tight_layout()
plt.show()

rf = model.named_steps["rf"]
importance = pd.DataFrame({
    "Feature": pollutant_cols,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)
display(importance)
importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()


## 10. Export Power BI-ready CSV with predictions

In [ ]:
powerbi = model_data.copy()
powerbi["predicted_aqi"] = np.nan
powerbi["prediction_error"] = np.nan
powerbi["absolute_error"] = np.nan
powerbi["residual"] = np.nan
powerbi["predicted_aqi_category"] = np.nan
powerbi["model_split"] = "Train"

test_idx = test_df.index
powerbi.loc[test_idx, "predicted_aqi"] = y_pred
powerbi.loc[test_idx, "prediction_error"] = test_df["aqi"].to_numpy() - y_pred
powerbi.loc[test_idx, "absolute_error"] = np.abs(test_df["aqi"].to_numpy() - y_pred)
powerbi.loc[test_idx, "residual"] = test_df["aqi"].to_numpy() - y_pred
powerbi.loc[test_idx, "predicted_aqi_category"] = [aqi_category(v) for v in y_pred]
powerbi.loc[test_idx, "model_split"] = "Test"

powerbi_file = OUTPUT_DIR / "mumbai_navi_mumbai_aqi_powerbi.csv"
powerbi.to_csv(powerbi_file, index=False)

# City/station coverage report is important for documenting the data limitation.
coverage_all.to_csv(OUTPUT_DIR / "station_coverage_report.csv", index=False)

print("Saved files:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", f.resolve())
print("\nPower BI rows/columns:", powerbi.shape)
display(powerbi.head())


## 11. Before presenting

- Use `latitude` and `longitude` as Decimal Number fields in Power BI and assign their data categories.
- Use `city` as a slicer, and `date`, `year`, `month_name`, `season`, and `aqi_category` for trends.
- Use only `model_split = Test` when visualizing Actual vs Predicted AQI and prediction error.
- Import `model_metrics.csv` for MAE/RMSE/R² cards and `feature_importance.csv` for the importance chart.
- Use `station_coverage_report.csv` to document which stations actually had observations.
- The current supplied historical dataset has no Navi Mumbai daily observations. To compare both cities over 2015–2020, add a second legitimate source with historical Navi Mumbai measurements and harmonize its pollutant units, dates, AQI definitions, and station IDs before training/evaluation.
- Some coordinate entries are approximate localities, not exact monitor points. Verify those before claiming precise spatial findings.